In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Nutzungsschätzung: unter ana Minute auf am Eagle r3-Prozessor (HINWEIS: Des is nur a Schätzung. Ihri Laufzeit ko variirn.)*

## Hintergrund {#Hintergrund}


Amplitudenverstärkung is a universeller Quantenalgorithmus oder a Unterroutine, de mia verwenda kenna, um a quadratische Beschleunigung gegenüber ana Handvoll klassischer Algorithmen z'erzieln. [Grovers Algorithmus](https://arxiv.org/abs/quant-ph/9605043) war da Erschte, wo diese Beschleunigung bei unstrukturierten Suchproblemen demonstriert hod. D'Formulierung von am Groverschen Suchproblem erfordert a Orakelfunktion, de oana oder mehrere Basiszuständ als de Zuständ markiert, wo mia findn woin, und an Verstärkungsschaltkreis, de d'Amplitude von de markierten Zuständ erhöht und folglich de verbleibenden Zuständ unterdrückt.

Do zeign mia, wia mia Grover-Orakel konstruiern und den [`grover_operator()`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.grover_operator) aus da Qiskit-Schaltkreisbibliothek verwendn, um einfach a Groversche Suchinstanz einzurichtn. Des Runtime `Sampler`-Primitiv ermöglicht de nahtlose Ausführung von Grover-Schaltkreisen.

## Anforderungen {#Anforderungen}


Schau vor'm Anfanga von diesem Tutorial, dass mia des Folgende installierd ham:

* Qiskit SDK v1.4 oder neier, mit [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)-Unterstützung
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.36 oder neier

## Setup {#Setup}

In [1]:
# Built-in modules
import math

# Imports from Qiskit
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, MCMTGate, ZGate
from qiskit.visualization import plot_distribution
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Imports from Qiskit Runtime
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler


def grover_oracle(marked_states):
    """Build a Grover oracle for multiple marked states

    Here we assume all input marked states have the same number of bits

    Parameters:
        marked_states (str or list): Marked states of oracle

    Returns:
        QuantumCircuit: Quantum circuit representing Grover oracle
    """
    if not isinstance(marked_states, list):
        marked_states = [marked_states]
    # Compute the number of qubits in circuit
    num_qubits = len(marked_states[0])

    qc = QuantumCircuit(num_qubits)
    # Mark each target state in the input list
    for target in marked_states:
        # Flip target bit-string to match Qiskit bit-ordering
        rev_target = target[::-1]
        # Find the indices of all the '0' elements in bit-string
        zero_inds = [
            ind
            for ind in range(num_qubits)
            if rev_target.startswith("0", ind)
        ]
        # Add a multi-controlled Z-gate with pre- and post-applied X-gates (open-controls)
        # where the target bit-string has a '0' entry
        if zero_inds:
            qc.x(zero_inds)
        qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)
        if zero_inds:
            qc.x(zero_inds)
    return qc

## Schritt 1: Klassische Eingaben auf a Quantenproblem abbildn {#Schritt-1-Klassische-Eingaben-auf-ein-Quantenproblem-abbilden}

Grovers Algorithmus braucht a [Orakel](/learning/courses/fundamentals-of-quantum-algorithms/grover-algorithm/introduction), de oana oder mehrere markierte Basiszuständ spezifiziert, wobei "markiert" an Zustand mit ana Phase von -1 bedeutet. A Controlled-Z-Gate oder sei mehrfach kontrollierte Verallgemeinerung über $N$ Qubits markiert de $2^{N}-1$-Zustand (`'1'`*$N$ Bit-String). Des Markiern von Basiszuständen mit oam oder mehreren `'0'` in da binären Darstellung erfordert des Anwendn von X-Gates auf de entsprechenden Qubits vor und nach'm Controlled-Z-Gate; des entspricht ana offenen Kontrolle auf diesem Qubit. Im nachfolgnden Code definiern mia a Orakel, de genau des macht und oana oder mehrere Eingabe-Basiszuständ markiert, de durch ihre Bit-String-Darstellung definiert san. Des `MCMT`-Gate wird verwendt, um des mehrfach kontrollierte Z-Gate z'implementiern.

In [2]:
marked_states = ["011", "100"]

oracle = grover_oracle(marked_states)
oracle.draw(output="mpl", style="iqp")

<Image src="../docs/images/tutorials/grovers-algorithm/extracted-outputs/c150298f-0.avif" alt="Output of the previous code cell" />

### Grover operator

The built-in Qiskit `grover_operator()` takes an oracle circuit and returns a circuit that is composed of the oracle circuit itself and a circuit that amplifies the states marked by the oracle.  Here, we use the `decompose()` method the circuit to see the gates within the operator:

In [3]:
grover_op = grover_operator(oracle)
grover_op.decompose().draw(output="mpl", style="iqp")

<Image src="../docs/images/tutorials/grovers-algorithm/extracted-outputs/283d5265-0.avif" alt="Output of the previous code cell" />

### Spezifische Grover-Instanz {#Spezifische-Grover-Instanz}

Weil mia jetzt d'Orakelfunktion ham, kenna mia a spezifische Instanz von da Groverschen Suche definiern. In diesem Beispü wern mia zwoa Berechnungszuständ aus de acht verfügbarn in am Drei-Qubit-Berechnungsraum markiern:

In [4]:
optimal_num_iterations = math.floor(
    math.pi
    / (4 * math.asin(math.sqrt(len(marked_states) / 2**grover_op.num_qubits)))
)

### Full Grover circuit

A complete Grover experiment starts with a Hadamard gate on each qubit; creating an even superposition of all computational basis states, followed the Grover operator (`grover_op`) repeated the optimal number of times.  Here we make use of the `QuantumCircuit.power(INT)` method to repeatedly apply the Grover operator.

In [5]:
qc = QuantumCircuit(grover_op.num_qubits)
# Create even superposition of all basis states
qc.h(range(grover_op.num_qubits))
# Apply Grover operator the optimal number of times
qc.compose(grover_op.power(optimal_num_iterations), inplace=True)
# Measure all qubits
qc.measure_all()
qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/tutorials/grovers-algorithm/extracted-outputs/4933ae44-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

For the small-scale simulation, we transpile the circuit without targeting specific hardware.

In [6]:
pm = generate_preset_pass_manager(optimization_level=3)
circuit_isa = pm.run(qc)
circuit_isa.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/grovers-algorithm/extracted-outputs/c4f67f35-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/grovers-algorithm/extracted-outputs/d3a26fc9-9090-4527-a749-a412661260b6-0.avif)

### Grover-Operator {#Grover-Operator}

Da eingebaute Qiskit `grover_operator()` nimmt an Orakel-Schaltkreis entgegn und gibt an Schaltkreis zruck, wo aus'm Orakel-Schaltkreis selber und am Schaltkreis besteht, de de vom Orakel markierten Zuständ verstärkt. Do verwendn mia d'`decompose()`-Methode von dem Schaltkreis, um de Gates innerhalb vom Operator z'segn:

In [7]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()
result = sampler.run([circuit_isa], shots=10_000).result()
dist = result[0].data.meas.get_counts()

![Output of the previous code cell](../docs/images/tutorials/grovers-algorithm/extracted-outputs/283d5265-0.avif)

Wiederholte Anwendungen von diesem `grover_op`-Schaltkreis verstärkn de markierten Zuständ und machns zu de wahrscheinlichsten Bit-Strings in da Ausgabeverteilung von dem Schaltkreis. Es gibt a optimale Anzahl solcher Anwendungen, de durch des Verhältnis von markierten Zuständen zur Gesamtzahl möglicher Berechnungszuständ bestimmt wird:

In [8]:
plot_distribution(dist)

<Image src="../docs/images/tutorials/grovers-algorithm/extracted-outputs/a5ef9913-0.avif" alt="Output of the previous code cell" />

### Vollständiger Grover-Schaltkreis {#Vollständiger-Grover-Schaltkreis}

A vollständiges Grover-Experiment fangt mit am Hadamard-Gate auf jedem Qubit oo; des erzeugt a gleichmäßige Überlagerung aller Berechnungsbasisstände, gefolgt vom Grover-Operator (`grover_op`), de d'optimale Anzahl von Moi widderholt wird. Do verwendn mia d'`QuantumCircuit.power(INT)`-Methode, um de Grover-Operator wiederholt anzwendn.

In [ ]:
# -------------------------Step 1-------------------------
marked_states = ["011", "100"]

oracle = grover_oracle(marked_states)
grover_op = grover_operator(oracle)

optimal_num_iterations = math.floor(
    math.pi
    / (4 * math.asin(math.sqrt(len(marked_states) / 2**grover_op.num_qubits)))
)

qc = QuantumCircuit(grover_op.num_qubits)
qc.h(range(grover_op.num_qubits))
qc.compose(grover_op.power(optimal_num_iterations), inplace=True)
qc.measure_all()

# -------------------------Step 2-------------------------
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
circuit_isa = pm.run(qc)

# -------------------------Step 3-------------------------
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10_000
sampler.options.environment.job_tags = ["TUT-GA"]
result = sampler.run([circuit_isa]).result()
dist = result[0].data.meas.get_counts()

# -------------------------Step 4-------------------------
plot_distribution(dist)

<Image src="../docs/images/tutorials/grovers-algorithm/extracted-outputs/be3c3d9e-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/grovers-algorithm/extracted-outputs/4933ae44-0.avif)

## Schritt 2: Problem für d'Ausführung auf Quantenhardware optimiern {#Schritt-2-Problem-für-die-Ausführung-auf-Quantenhardware-optimieren}

In [10]:
import matplotlib.pyplot as plt

num_qubits_list = list(range(3, 10))
two_q_depths = []
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
for n in num_qubits_list:
    # Mark a single state for simplicity
    marked = ["1" * n]
    oracle_n = grover_oracle(marked)
    grover_op_n = grover_operator(oracle_n)

    # Optimal number of iterations
    num_iters = math.floor(
        math.pi / (4 * math.asin(math.sqrt(len(marked) / 2**n)))
    )

    # Build the full Grover circuit
    qc_n = QuantumCircuit(n)
    qc_n.h(range(n))
    qc_n.compose(grover_op_n.power(num_iters), inplace=True)
    qc_n.measure_all()

    # Transpile to a basis gate set and count 2Q depth
    pm_n = generate_preset_pass_manager(backend=backend, optimization_level=3)
    qc_transpiled = pm_n.run(qc_n)

    # Compute depth restricted to 2-qubit operations
    depth_2q = qc_transpiled.depth(lambda x: x.operation.num_qubits == 2)

    two_q_depths.append(depth_2q)
    print(f"n={n}: optimal_iters={num_iters}, 2Q depth={depth_2q}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    num_qubits_list,
    two_q_depths,
    "o-",
    linewidth=2,
    markersize=8,
    color="#6929C4",
)
ax.set_xlabel("Number of qubits", fontsize=13)
ax.set_ylabel("Two-qubit gate depth", fontsize=13)
ax.set_title("Grover's algorithm: 2Q depth scaling", fontsize=14)
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
ax.set_xticks(num_qubits_list)
plt.tight_layout()
plt.show()

n=3: optimal_iters=2, 2Q depth=39
n=4: optimal_iters=3, 2Q depth=111
n=5: optimal_iters=4, 2Q depth=466
n=6: optimal_iters=6, 2Q depth=1646
n=7: optimal_iters=8, 2Q depth=3550
n=8: optimal_iters=12, 2Q depth=7989
n=9: optimal_iters=17, 2Q depth=14824


<Image src="../docs/images/tutorials/grovers-algorithm/extracted-outputs/abc6b43c-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/grovers-algorithm/extracted-outputs/c9a3020e-0.avif)

## Schritt 3: Ausführn mit Qiskit-Primitiven {#Schritt-3-Ausführen-mit-Qiskit-Primitiven}


Amplitudenverstärkung is a Sampling-Problem, des für d'Ausführung mit'm [`Sampler`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)-Runtime-Primitiv geeignet is.

Beachts, dass d'`run()`-Methode vom [Qiskit Runtime `SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) a Iterable von `primitive unified blocks (PUBs)` akzeptiert. Für de Sampler is jedes PUB a Iterable im Format `(circuit, parameter_values)`. Mindestens akzeptiert er aber a Listn von Quantenschaltkreis(en).